In [ ]:
import numpy as np
import torch
from synthetic import simulate_var
from models.cmlp import cMLP,train_model_ista,train_model_adam
from copy import deepcopy
from tqdm import tqdm
from pathlib import Path
import pickle
from datetime import datetime

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 参数设置
HUP_LIST = ['116']    # 受试者ID列表
type = 'ictal'        # 任务类型（发作期）
# type = 'interictal'        # 任务类型（发作期）
fs = 256
start = -80                   # 发作起始时间（秒）
end = 80                  # 发作结束时间（秒）

DATA_PATH = Path(r"G:\DataSet\HUP_iEEG_python")  # 数据根目录
RESULT_PATH = Path(r'G:\RESULT\cmlp')   # 结果储存根目录

In [ ]:
for sub in range (len(HUP_LIST)):
    for run in [1]:
        subject_id = 'HUP' + HUP_LIST[sub]
        data_dir = DATA_PATH / subject_id
        filename = f"sub-{subject_id}_task-{type}_run-{run:02d}_{start}-{end}_{fs}Hz.npz" # 测试修改
        data_path = data_dir / filename

        loaded_data = np.load(data_path, allow_pickle=True)

        result_dir = RESULT_PATH / subject_id / f"{type}_RUN{run:02d}"
        result_dir.mkdir(parents=True, exist_ok=True)
        
        data_dict = {key: loaded_data[key] for key in loaded_data.files}
        data_x = data_dict['data']

In [ ]:
data = data_dict['data']
fs = int(data_dict['fs'])  # 采样率
channel_names = data_dict['channel_names']

win_len_sec = 5       # 滑动窗口长度（秒）
step_len_sec = 5      # 滑动步长（秒）


_WINDOWS_SIZE = win_len_sec * fs
_WINDOWS_STEP = step_len_sec * fs

n_channels = data.shape[0]
n_samples = data.shape[1]
num_windows = (n_samples - _WINDOWS_SIZE) // _WINDOWS_STEP + 1


In [ ]:
print(f'数据信息:\tHUP{HUP_LIST[sub]}')
print(f'dataShape:\t{data.shape}')
print(f'fs:\t{fs}')
print(f'通道数:\t {n_channels}')
print(f'数据路径:\t{data_path}') 
print(f'窗口大小:\t{_WINDOWS_SIZE}')
print(f'窗口步长:\t{_WINDOWS_STEP}')
print(f'窗口数:\t{num_windows}')


In [ ]:
data_x= torch.tensor(data_x[np.newaxis], dtype=torch.float32, device=device)
data_x = data_x.transpose(2, 1)
channels = [0, 4, 8, 12, 14, 18, 22, 26, 30, 34, 38, 42, 46]
X_select = data_x[:, :, channels]

In [ ]:
cmlp = cMLP(num_series=X_select.shape[-1],
            lag=5,
            hidden=[100])

In [ ]:
# train_loss_list = train_model_ista(
#     cmlp, data_x, lam=0.002, lam_ridge=1e-2, lr=5e-2, penalty='H', max_iter=50000,
#     check_every=100)

In [ ]:
dynamic_gc_list = []

for start in tqdm(range(0, data_x.shape[1] - _WINDOWS_SIZE, _WINDOWS_STEP), desc="Dynamic GC Windows"):
  X_window = X_select[:, start:start + _WINDOWS_SIZE,:]
  print(torch.isnan(X_window).any())
  print(torch.isinf(X_window).any())
  print(X_window.max(), X_window.min())
  cmlp_dynamic = deepcopy(cmlp)
  train_loss_list = train_model_ista(
    cmlp_dynamic, X_window, lam=0.002, lam_ridge=1e-4, lr=5e-2, penalty='H', max_iter=50000,
    check_every=100)
  dynamic_gc_list.append(cmlp_dynamic.GC(threshold=False))


In [ ]:
# 假设你只想取第 2, 5, 10 个窗口
selected_windows = [2, 17]

for idx in tqdm(selected_windows, desc="Selected GC Windows"):
    start = idx * _WINDOWS_STEP
    X_window = X_select[:, start:start + _WINDOWS_SIZE, :]

    print(torch.isnan(X_window).any())
    print(torch.isinf(X_window).any())
    print(X_window.max(), X_window.min())

    cmlp_dynamic = deepcopy(cmlp)
    train_loss_list = train_model_ista(
        cmlp_dynamic, X_window, lam=0.002, lam_ridge=1e-4, lr=5e-2, 
        penalty='H', max_iter=50000, check_every=100
    )
    dynamic_gc_list.append(cmlp_dynamic.GC(threshold=False))


In [ ]:
timeStamp = datetime.now().strftime("%Y%m%d-%H%M%S")
filename = RESULT_PATH /  f"dynamic_gc_list_{timeStamp}.pkl"

with open(filename, "wb") as f:
    pickle.dump(dynamic_gc_list, f)

print(f"保存地址：{filename}, 包含时间窗口：{len(dynamic_gc_list)}")